# Pipeline de Producción: Ingeniería de Features para ML
## Módulo 01: Ingestión y Transformación desde la Capa Gold

Este componente del ecosistema analítico en **Microsoft Fabric** representa el punto de entrada para el despliegue del modelo en producción. El objetivo fundamental es realizar la ingesta masiva de datos transaccionales optimizados desde el Lakehouse y aplicar las transformaciones necesarias para construir el dataset ML-Ready.

---

## 🏗️ Arquitectura de Ingestión y Gobierno de Datos

El pipeline consume los datos estructurados directamente de la capa **Gold** del Lakehouse. Al operar sobre esta capa, el sistema se asegura de que la información ya ha pasado por procesos previos de limpieza (Bronze) y transformaciones/reglas de negocio intermedias (Silver), garantizando la consistencia del esquema de entrada.

### 🔌 Carga Eficiente mediante PySpark
Para la lectura de la tabla de hechos, el código implementa la interfaz nativa `spark.read.table`. Esto aprovecha el motor distribuido de Synapse para segmentar el volumen transaccional en memoria distribuida, optimizando los tiempos de transferencia antes de la ejecución del algoritmo de Machine Learning.


In [1]:
# ========================================================
# 01. LECTURA DE DATOS DESDE LA CAPA GOLD
# ========================================================
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import when, col, count, isnan, isnull

# Ingestión nativa desde la tabla de hechos del catálogo relacional
# NOTA: Se unifica el nombre de tabla al formato estándar del catálogo Fabric
df = spark.read.table("gold.dbo.hechos_ventas")

print(f"✅ Datos cargados correctamente desde gold.dbo.hechos_ventas")
print(f"📊 Cantidad total de registros a procesar: {df.count():,}")

display(df)

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 3, Finished, Available, Finished, False)

✅ Datos cargados correctamente desde gold.dbo.hechos_ventas
📊 Cantidad total de registros a procesar: 1,000


SynapseWidget(Synapse.DataFrame, bcc3cce4-c8c5-4844-921c-382ca4d6779a)

## 🔍 Validación de Calidad: Verificación de Nulos

Antes de aplicar cualquier transformación, es indispensable auditar la integridad de las columnas clave. La presencia de valores nulos en `age`, `id_genero` o `total_amount` puede producir resultados silenciosamente incorrectos en las etapas de feature engineering.

**Columnas críticas a validar:**
- `total_amount` → define la variable objetivo `nivel_venta`
- `age` → alimenta el binning `grupo_edad`
- `id_genero` → fuente del One-Hot Encoding de género

In [2]:
# ========================================================
# 02. VALIDACIÓN DE NULOS EN COLUMNAS CLAVE
# ========================================================
columnas_criticas = ["total_amount", "age", "id_genero", "customer_id", "id_categoria"]

print("📋 Reporte de valores nulos en columnas clave:")
print("-" * 45)

nulos_encontrados = False
for col_name in columnas_criticas:
    n_nulos = df.filter(isnull(col(col_name)) | isnan(col(col_name)) if col_name != "id_genero" else isnull(col(col_name))).count()
    estado = "❌ REQUIERE ATENCIÓN" if n_nulos > 0 else "✅ Sin nulos"
    print(f"  {col_name:<20}: {n_nulos:>6,} nulos  {estado}")
    if n_nulos > 0:
        nulos_encontrados = True

print("-" * 45)
if nulos_encontrados:
    print("⚠️  Se detectaron nulos. Evaluar estrategia de imputación antes de continuar.")
else:
    print("✅ Todas las columnas clave están completas. Proceso puede continuar.")

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 4, Finished, Available, Finished, False)

📋 Reporte de valores nulos en columnas clave:
---------------------------------------------
  total_amount        :      0 nulos  ✅ Sin nulos
  age                 :      0 nulos  ✅ Sin nulos
  id_genero           :      0 nulos  ✅ Sin nulos
  customer_id         :      0 nulos  ✅ Sin nulos
  id_categoria        :      0 nulos  ✅ Sin nulos
---------------------------------------------
✅ Todas las columnas clave están completas. Proceso puede continuar.


## 🎯 Ingeniería de la Variable Objetivo (Label Engineering)

Dado que los registros originales contienen métricas transaccionales continuas, el pipeline requiere una fase de transformación para estructurar el problema bajo el enfoque de **Clasificación Binaria**.

### 🔹 Regla de Negocio y Segmentación Algorítmica
Para convertir el flujo transaccional en clases discretas, se implementa una lógica condicional basada en el volumen financiero acumulado de la transacción (`total_amount`). Esta regla mapea el comportamiento del cliente en dos categorías estratégicas:

* **Clase 0: Venta Normal** → Transacciones con `total_amount < $1000`. Representa el flujo operativo estándar.
* **Clase 1: Venta VIP** → Transacciones con `total_amount >= $1000`. Define el segmento de alto impacto financiero.

> **Justificación del umbral $1000:** El corte se apoya en el análisis estadístico de `total_amount` (ver celda siguiente). Se recomienda revisar la mediana y el percentil 75 para validar que el umbral capture efectivamente el segmento de alto valor sin generar desbalance severo de clases.


In [3]:
# ========================================================
# 03. ESTADÍSTICAS DESCRIPTIVAS PARA JUSTIFICAR EL UMBRAL
# ========================================================
from pyspark.sql.functions import percentile_approx, avg, stddev, min as spark_min, max as spark_max

print("📊 Estadísticas descriptivas de total_amount:")
df.select(
    spark_min("total_amount").alias("min"),
    percentile_approx("total_amount", 0.25).alias("p25"),
    percentile_approx("total_amount", 0.50).alias("mediana"),
    avg("total_amount").alias("media"),
    percentile_approx("total_amount", 0.75).alias("p75"),
    spark_max("total_amount").alias("max")
).show()

# ========================================================
# 04. CREACIÓN DE LA VARIABLE OBJETIVO: nivel_venta
# ========================================================
df_features = df.withColumn(
    "nivel_venta",
    when(col("total_amount") >= 1000, 1).otherwise(0)
)

# Verificar distribución de clases (detectar posible desbalance)
print("\n📊 Distribución de clases en nivel_venta:")
df_features.groupBy("nivel_venta").count().orderBy("nivel_venta").show()

print("✅ Variable objetivo 'nivel_venta' creada correctamente.")

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 5, Finished, Available, Finished, False)

📊 Estadísticas descriptivas de total_amount:
+----+----+-------+-----+-----+------+
| min| p25|mediana|media|  p75|   max|
+----+----+-------+-----+-----+------+
|25.0|60.0|  120.0|456.0|900.0|2000.0|
+----+----+-------+-----+-----+------+


📊 Distribución de clases en nivel_venta:
+-----------+-----+
|nivel_venta|count|
+-----------+-----+
|          0|  798|
|          1|  202|
+-----------+-----+

✅ Variable objetivo 'nivel_venta' creada correctamente.


## 🔢 Codificación de Variables Categóricas (Manual One-Hot Encoding)

La columna `id_genero` está almacenada en la capa Gold como un atributo categórico. Los modelos lineales son incapaces de interpretar directamente variables nominales sin orden jerárquico, por lo que se aplica **One-Hot Encoding** manual:

* **`gender_M`**: Valor `1` si `id_genero == "2"` (Masculino); `0` en caso contrario.
* **`gender_F`**: Valor `1` si `id_genero == "1"` (Femenino); `0` en caso contrario.

La transformación emplea `.cast(IntegerType())` para garantizar enteros de 32 bits, el formato requerido por el vectorizador del modelo.

In [4]:
# ========================================================
# 05. ONE-HOT ENCODING DE GÉNERO
# ========================================================
from pyspark.sql.functions import when, col
from pyspark.sql.types import IntegerType

df_features = df_features.withColumn(
    "gender_M", when(col("id_genero") == "2", 1).otherwise(0).cast(IntegerType())
)
df_features = df_features.withColumn(
    "gender_F", when(col("id_genero") == "1", 1).otherwise(0).cast(IntegerType())
)

print("✅ One-Hot Encoding de género aplicado.")

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 6, Finished, Available, Finished, False)

✅ One-Hot Encoding de género aplicado.


## 📊 Discretización de Variables Continuas (Feature Binning)

Para capturar dinámicas de comportamiento asociadas al ciclo de vida del consumidor, el pipeline implementa una fase de discretización sobre `age`. En lugar de variaciones anuales aisladas, los modelos capturan mejor **etapas de vida generacionales**:

* **`joven`**: `age < 30` — segmento emergente.
* **`adulto`**: `30 <= age < 50` — mayor estabilidad financiera.
* **`adulto_mayor`**: `age >= 50` — segmento maduro del mercado.

In [5]:
# ========================================================
# 06. BINNING DE EDAD
# ========================================================
df_features = df_features.withColumn(
    "grupo_edad",
    when(col("age") < 30, "joven")
    .when((col("age") >= 30) & (col("age") < 50), "adulto")
    .otherwise("adulto_mayor")
)

print("✅ Discretización de edad aplicada.")

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 7, Finished, Available, Finished, False)

✅ Discretización de edad aplicada.


## 🔍 Auditoría Visual y Validación de Transformaciones

Se realiza una inspección cruzada de las variables transformadas para verificar:

1. **Integridad de `nivel_venta`**: Transacciones VIP (1) y Normales (0) correctamente clasificadas.
2. **Consistencia del OHE de género**: Sin colisiones (una fila no puede tener `gender_M=1` y `gender_F=1` simultáneamente).
3. **Correcta discretización de `grupo_edad`**: Fronteras en 30 y 50 años correctamente aplicadas.

In [6]:
# ========================================================
# 07. AUDITORÍA VISUAL DE TRANSFORMACIONES
# ========================================================
display(df_features.select("total_amount", "nivel_venta", "id_genero", "gender_M", "gender_F", "age", "grupo_edad"))

# Validación de exclusión mutua en OHE de género
colisiones = df_features.filter((col("gender_M") == 1) & (col("gender_F") == 1)).count()
print(f"\n🔎 Colisiones en OHE género (esperado 0): {colisiones}")
if colisiones == 0:
    print("✅ Exclusión mutua verificada correctamente.")
else:
    print("❌ Se detectaron colisiones. Revisar lógica de codificación.")

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e02dfde2-7a2f-4280-96d1-cccf82713828)


🔎 Colisiones en OHE género (esperado 0): 0
✅ Exclusión mutua verificada correctamente.


## 📦 Consolidación del Dataset ML-Ready

Se construye el dataset final con las columnas necesarias, organizadas en tres categorías:

1. **Identificadores de trazabilidad**: `customer_id`, `id_categoria`, `id_genero`.
   > ⚠️ **IMPORTANTE**: Estas columnas se incluyen **únicamente para trazabilidad y auditoría**. **NO deben pasarse al vectorizador** (`VectorAssembler`) en el notebook de entrenamiento (`02_model_training`). Excluirlas explícitamente del array de `inputCols`.

2. **Métricas continuas y demográficas**: `quantity`, `price_per_unit`, `total_amount`, `age`.

3. **Features derivadas (features de entrenamiento)**: `grupo_edad`, `gender_F`, `gender_M`, `nivel_venta` (variable objetivo).

In [7]:
# ========================================================
# 08. SELECCIÓN Y CONSOLIDACIÓN DEL DATASET FINAL
# ========================================================
feature_cols = [
    # ── Identificadores de trazabilidad (NO usar en VectorAssembler) ──
    "customer_id", "id_categoria", "id_genero",
    # ── Métricas continuas originales ────────────────────────────────
    "quantity", "price_per_unit", "total_amount", "age",
    # ── Features derivadas + variable objetivo ────────────────────────
    "grupo_edad", "gender_F", "gender_M", "nivel_venta"
]

df_gold = df_features.select(feature_cols)

print(f"✅ Columnas seleccionadas: {len(feature_cols)}")
print(f"✅ Registros finales: {df_gold.count():,}")
print(f"\n📌 Recordatorio: customer_id, id_categoria e id_genero son de trazabilidad.")
print(f"   NO incluirlos en el VectorAssembler del notebook 02_model_training.")

display(df_gold.limit(10))

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 9, Finished, Available, Finished, False)

✅ Columnas seleccionadas: 11
✅ Registros finales: 1,000

📌 Recordatorio: customer_id, id_categoria e id_genero son de trazabilidad.
   NO incluirlos en el VectorAssembler del notebook 02_model_training.


SynapseWidget(Synapse.DataFrame, 41f8b355-19b5-459f-b523-374f1c96fc83)

## 💾 Persistencia en el Lakehouse en Formato Delta Lake

El dataset transformado se persiste en el Lakehouse bajo el estándar **Delta Lake**, que garantiza:

1. **Transacciones ACID**: Operaciones atómicas y consistentes — si el clúster falla a mitad del proceso, la operación se revierte por completo.
2. **Time Travel**: Permite auditar versiones previas del dataset para reproducibilidad de experimentos.
3. **Overwrite limpio**: El modo `overwrite` reemplaza la ejecución anterior sin interrumpir consultas activas en Power BI u otros consumidores.

> **Nombre de tabla**: Se usa `gold_features_clasificacion` siguiendo la convención snake_case descriptiva del gobierno de datos del proyecto.

In [8]:
# ========================================================
# 09. ESCRITURA EN DELTA LAKE
# ========================================================
# Nombre descriptivo en snake_case (gov. de datos: reemplaza 'tabla_gold')
NOMBRE_TABLA = "gold_features_clasificacion"

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(NOMBRE_TABLA)

print(f"✅ Dataset ML-Ready persistido en: {NOMBRE_TABLA}")
print(f"📦 Formato: Delta Lake | Modo: overwrite")
print(f"🔗 Próximo paso: Consumir '{NOMBRE_TABLA}' en 02_model_training.ipynb")

StatementMeta(, a34c2260-1a04-4443-883a-f2dcc16d5115, 10, Finished, Available, Finished, False)

✅ Dataset ML-Ready persistido en: gold_features_clasificacion
📦 Formato: Delta Lake | Modo: overwrite
🔗 Próximo paso: Consumir 'gold_features_clasificacion' en 02_model_training.ipynb
